# CampusClaim — Gradio demo (no training)

Reload the **saved LoRA adapters** from Google Drive and launch the receipt → claim app with the **Hacktoberfest UI** from `app/gradio_app.py`.

**Runtime:** Runtime → Change runtime type → **T4 GPU**

**Run order:** install → mount Drive → load base model → attach adapters → helpers → Gradio.

Skip the full `campusclaim_3b.ipynb` training notebook; this file only serves the demo.


## 1. Install (2–4 minutes)


In [ ]:
%%capture
# Same stack as campusclaim_3b.ipynb
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
!pip install "gradio>=6,<7" pandas


## 2. Mount Google Drive

Upload folder should contain `adapter_config.json` and `adapter_model.safetensors`.

Default path: `MyDrive/campusclaim/campusclaim_gemma4_lora` — change `ADAPTER` if yours differs.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ADAPTER = Path("/content/drive/MyDrive/campusclaim/campusclaim_gemma4_lora")

# If you uploaded a zip, uncomment:
# !unzip -qo /content/drive/MyDrive/campusclaim/campusclaim_gemma4_lora.zip -d /content/drive/MyDrive/campusclaim

assert (ADAPTER / "adapter_config.json").is_file(), (
    f"Missing adapter_config.json under {ADAPTER}. "
    f"Contents: {list(ADAPTER.parent.glob('*')) if ADAPTER.parent.exists() else 'parent missing'}"
)
assert (ADAPTER / "adapter_model.safetensors").is_file(), f"Missing weights in {ADAPTER}"
print("Adapter OK:", ADAPTER)
print("Files:", sorted(p.name for p in ADAPTER.iterdir()))


## 3. Load base Gemma 4 E2B (4-bit)


In [ ]:
import gc
import torch
from unsloth import FastVisionModel, get_chat_template

MODEL_NAME = "unsloth/gemma-4-E2B-it"
INSTRUCTION = "Extract the company, date, address and total from this receipt. Reply with JSON only."

model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
processor = get_chat_template(processor, "gemma-4")
gc.collect()
torch.cuda.empty_cache()
print("Loaded", MODEL_NAME)
print("device:", next(model.parameters()).device)
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")


## 4. Attach saved LoRA adapters (no training)


In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(model, str(ADAPTER))
model.eval()
FastVisionModel.for_inference(model)
print("Adapters attached from", ADAPTER)


## 5. Inference helpers


In [ ]:
import re
import json

MAX_IMAGE_SIDE = 1024

def prepare_image(image):
    image = image.convert("RGB")
    image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
    return image

def parse_json(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
        return value if isinstance(value, dict) else None
    except json.JSONDecodeError:
        return None

def generate_from_image(image, instruction=INSTRUCTION, max_new_tokens=128):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        prepare_image(image), input_text, add_special_tokens=False, return_tensors="pt"
    ).to("cuda")
    with torch.inference_mode():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False
        )
    return processor.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()

print("Helpers ready")


## 6. Gradio demo (Hacktoberfest UI)

Pulls the updated UI from the repo (`app/gradio_app.py`: hero, theme, CSS, claim draft, webcam unmirrored).

After launch, copy the public `*.gradio.live` URL into the MLH live-demo field.


In [ ]:
# Fresh UI source from the repo (dev branch)
!git clone --depth 1 -b dev https://github.com/sank8-2/campusclaim.git /content/campusclaim-ui 2>/dev/null || \
  (cd /content/campusclaim-ui && git fetch --depth 1 origin dev && git checkout -B dev origin/dev)

import sys
sys.path.insert(0, "/content/campusclaim-ui")

from app.gradio_app import (
    APP_CSS,
    build_app,
    build_theme,
    make_dual_model_extractors,
)

def infer_json(_model, image):
    """Notebook generate_from_image already closes over the loaded Peft model."""
    text = generate_from_image(image)
    return parse_json(text) or {}

service = make_dual_model_extractors(model, infer_json)
demo = build_app(service)
demo.launch(share=True, theme=build_theme(), css=APP_CSS)
